<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Erste Agenten mit LangChain
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M03-Erste-Agenten"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---

In diesem Modul wird ein **erster vollständiger KI-Agenten** mit LangChain erstellt.

Ein Agent verbindet drei Komponenten:

| Komponente | Rolle | Beispiel |
|---|---|---|
| **LLM** | Entscheidet, welche Tools gebraucht werden | `gpt-4o-mini` |
| **Tools** | Führen konkrete Aktionen aus | `celsius_nach_fahrenheit()` |
| **System-Prompt** | Definiert Rolle und Verhalten | "Du bist ein hilfreicher Assistent..." |

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(BASELINE, temperature=0.0)

# 2 | create_agent() – Moderne Agent-API
---

`create_agent()` ist die standardisierte API für ReAct-Agenten in LangChain 1.0+.

**Die wichtigsten Parameter:**

| Parameter | Typ | Beschreibung |
|---|---|---|
| `model` | `str` oder LLM | Welches LLM steuert den Agenten? |
| `tools` | `list[Tool]` | Welche Werkzeuge stehen zur Verfügung? |
| `system_prompt` | `str` | Wie soll der Agent auftreten und antworten? |

**Was intern passiert (TAO-Zyklus):**
1. Agent sendet Nutzeranfrage + Tool-Schemas an das LLM
2. LLM entscheidet: direkte Antwort oder Tool-Call?
3. Bei Tool-Call: Tool ausführen, Ergebnis zurückliefern
4. Zyklus wiederholen bis Antwort vollständig

In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>

diagram = '''
flowchart LR
    LLM["LLM\ngpt-4o-mini"]
    TOOLS["Tools\ncelsius_nach_fahrenheit\nist_buerozeit"]
    SYS["System-Prompt\nRolle & Verhalten"]
    AGENT["create_agent()"]
    RUN["agent.invoke(...)"]

    LLM --> AGENT
    TOOLS --> AGENT
    SYS --> AGENT
    AGENT --> RUN
'''

mermaid(diagram, width=600)

In [ ]:
# 2.2 Agent zusammensetzen mit create_agent()
from langchain.agents import create_agent
from langchain_core.tools import tool

# Tools definieren
@tool
def celsius_nach_fahrenheit(temperatur: float) -> float:
    """Rechnet eine Temperatur von Celsius in Fahrenheit um. Gibt den Fahrenheit-Wert zurueck."""
    return round(temperatur * 9 / 5 + 32, 2)

@tool
def ist_buerozeit(stunde: int) -> str:
    """Prueft ob eine Stunde des Tages (0-23) in die typische Buerozeit faellt (9-17 Uhr).
    Gibt 'Buerozeit' oder 'Ausserhalb Buerozeit' zurueck."""
    return "Buerozeit" if 9 <= stunde <= 17 else "Ausserhalb Buerozeit"

system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_agent_system_prompt.md", mode="S")

# Agent zusammensetzen
agent = create_agent(
    model=BASELINE,
    tools=[celsius_nach_fahrenheit, ist_buerozeit],
    system_prompt=system_prompt,
)

mprint("**Agent bereit** mit folgenden Tools:")
for t in [celsius_nach_fahrenheit, ist_buerozeit]:
    mprint(f"- `{t.name}`: {t.description}")


# 3 | LCEL: Der Pipe-Operator
---

Der **Pipe-Operator `|`** verbindet LangChain-Bausteine zu einer Chain — dasselbe Prinzip steckt im Innern von `create_agent()`.

**Die drei Grundbausteine:**

| Baustein | Klasse | Aufgabe |
|---|---|---|
| **Prompt** | `ChatPromptTemplate` | Platzhalter füllen → Nachricht |
| **LLM** | `init_chat_model()` | Nachricht → AI-Antwort |
| **Parser** | `StrOutputParser()` | AIMessage → reiner Text |

```
chain = prompt | llm | parser
         ↓        ↓       ↓
      Eingabe → Denken → Ausgabe
```

> 💡 `create_agent()` baut auf genau diesem Prinzip auf — und erweitert es um Tool-Calls und den TAO-Zyklus.
> Chains und Agenten sind kein Gegensatz: **Agenten sind Chains mit Entscheidungslogik.**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_tpl = ChatPromptTemplate.from_template("Erkläre {thema} in einem Satz.")
parser     = StrOutputParser()

# Pipe-Operator verbindet die drei Bausteine
chain = prompt_tpl | llm | parser

antwort = chain.invoke({"thema": "KI-Agenten"})
mprint(f"**Chain-Antwort:** {antwort}")
mprint("\n---\n")
mprint("✅ Das ist die gleiche Struktur, die `create_agent()` intern verwendet — erweitert um Tool-Calls.")

# 4 | Erster Agent mit 2 Tools
---



Jetzt setzen wir alles zusammen: LLM, Tools und System-Prompt ergeben einen funktionierenden Agenten.

Der Agent entscheidet selbst, ob er ein Tool aufruft oder direkt antwortet – je nachdem, was die Anfrage erfordert.

In [ ]:
ergebnis = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Sind 21 Grad Celsius angenehm? Und: Es ist gerade 15 Uhr – ist das Bürozeit?"
    }]
})

mprint("## Agent-Antwort")
mprint(ergebnis["messages"][-1].content)

In [ ]:
mprint("## Verwendete Tool-Calls")
tools_gefunden = False
for msg in ergebnis["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tools_gefunden = True
        for tc in msg.tool_calls:
            mprint(f"- `{tc['name']}({tc['args']})`")
if not tools_gefunden:
    mprint("*Kein Tool-Call – Agent hat direkt geantwortet.*")

# 5 | Agent-Verhalten testen
---



Testen Sie den Agenten mit verschiedenen Anfragen und beobachten Sie:
- Wann ruft er ein Tool auf?
- Wann antwortet er direkt ohne Tool?
- Was passiert bei Fragen, die mehrere Tools erfordern?

Diese Beobachtungen helfen, den System-Prompt gezielt zu optimieren.

In [ ]:
test_fragen = [
    ("Tool erwartet",  "Was sind 5 Grad Celsius in Fahrenheit?"),
    ("Tool erwartet",  "Ist 22 Uhr noch Bürozeit?"),
    ("Kein Tool",      "Erkläre mir kurz, was ein KI-Agent ist."),
    ("Beide Tools",    "Es ist 10 Uhr. Brauche ich eine Jacke bei 3 Grad Celsius?"),
]

for erwartung, frage in test_fragen:
    result = agent.invoke({"messages": [{"role": "user", "content": frage}]})

    tools_genutzt = [
        tc["name"]
        for msg in result["messages"]
        if hasattr(msg, "tool_calls") and msg.tool_calls
        for tc in msg.tool_calls
    ]

    mprint(f"\n---\n**Frage [{erwartung}]:** {frage}")
    mprint(f"**Antwort:** {result['messages'][-1].content}")
    if tools_genutzt:
        mprint(f"**Tools genutzt:** `{'`, `'.join(tools_genutzt)}`")
    else:
        mprint("**Kein Tool genutzt** – direkte LLM-Antwort")

# 6 | LangSmith: Agent-Trace
---



LangSmith visualisiert den vollständigen TAO-Zyklus eines Agenten:
- **Thought:** Welche Tool-Calls hat das LLM geplant?
- **Action:** Welche Parameter wurden übergeben?
- **Observation:** Was hat das Tool zurückgeliefert?

**Im Trace sehen Sie** den genauen Entscheidungsweg des Agenten – unschätzbar für Debugging und Optimierung des System-Prompts.

Das Tracing ist bereits in der Setup-Cell konfiguriert (`LANGSMITH_TRACING`, `LANGSMITH_ENDPOINT`, Projekt `M03-Erste-Agenten`).

In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M03_Kap5_AgentTrace",
    "tags":     ["M03", "agent", "langsmith"]
}

# 2. with_config() anwenden, dann invoke()
trace_result = agent.with_config(**run_cfg).invoke({
    "messages": [{
        "role": "user",
        "content": "Wie viel Fahrenheit sind 100 Grad Celsius, und ist 16 Uhr Bürozeit?"
    }]
})

mprint("## Trace-Ergebnis")
mprint('---')
mprint(trace_result["messages"][-1].content)

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M03-Erste-Agenten", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.




<p><font color='black' size="5">
Erster eigener Agent
</font></p>

Erstellen Sie einen Agenten für einen Prozess aus Ihrem Arbeitsumfeld – nutzen Sie Ihre Tools aus *Tool Use & Function Calling* oder definieren Sie neue.

**Grundlagen**
1. Ein eigenes Tool mit dem `@tool`-Decorator definieren (Thema frei waehlbar, z. B. Einheitenrechner, Text-Utils).
2. Einen Agenten mit `create_agent()` und diesem Tool erstellen.
3. Eine einfache Testfrage stellen, bei der der Agent das Tool nutzt, und die Ausgabe zeigen.

Den Agenten in der Variable `mein_agent` speichern.


**Aufbau**
1. Zwei eigene Tools definieren (aus unterschiedlichen Bereichen, z. B. Rechnen + Text).
2. Einen System-Prompt schreiben, der die Rolle des Agenten klar beschreibt.
3. Den Agenten mit beiden Tools und dem eigenen System-Prompt erstellen.
4. Drei Testfragen stellen – davon mindestens 1 Frage, bei der der Agent kein Tool benoetigt.
5. Kurz notieren, wann der Agent Tools nutzt und wann nicht.

Agenten in `mein_agent`, Tools in `meine_tools` speichern.


**Vertiefung**
1. Ein drittes Tool hinzufügen.
2. 2 Anfragen stellen, bei denen der Agent selbst entscheidet, welches der 3 Tools er waehlt.
3. Den System-Prompt anpassen (z. B. staerker restriktiv oder ausfuehrlicher) und in 2-3 Saetzen beschreiben, wie sich das Verhalten aendert.
4. LangSmith-Tracing aktivieren und fuer einen Agenten-Lauf beschreiben, welche TAO-Schritte im Trace sichtbar sind.

Die Beobachtung zum System-Prompt in `prompt_beobachtung` (String) speichern.


<p><font color='black' size="5">
🔍 Selbstcheck mit `assert`
</font></p>

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Schreiben Sie Ihre Lösung in die Zellen über diesem Block
2. Speichern Sie Ihren Agenten in der Variable **`mein_agent`** (oder passen Sie den Namen in der Selbstcheck-Zelle an)
3. Führen Sie die Zelle unten aus — ein `✅` zeigt: Mindestanforderungen erfüllt


<p><font color='black' size="5">
✅ Selbstcheck
</font></p>

In [ ]:
# ► Speichere deinen Agenten in der Variable 'mein_agent' – oder passe die erste Zeile an.

_ag = mein_agent  # ← Variablennamen anpassen

assert hasattr(_ag, "invoke"), \
    "❌ Kein invoke() – wurde der Agent mit create_agent() erstellt?"

_r = _ag.invoke({"messages": [{"role": "user", "content": "Was kannst du für mich tun?"}]})
_antwort = _r["messages"][-1].content
assert len(_antwort) > 10, \
    f"❌ Antwort zu kurz ({len(_antwort)} Z.) – prüfe System-Prompt und Tools"

print(f"✅ Agent antwortet · {len(_antwort)} Zeichen")
print(f"   {_antwort[:120]}{'...' if len(_antwort) > 120 else ''}")
